In [0]:
# CELL 1
%pip install sentence-transformers faiss-cpu --quiet
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# CELL 2 — Load chunks
import json, os
import numpy as np

df_chunks = spark.read.table("sarkarimitracatalog.sarkaridatabase.scheme_chunks")

rows = df_chunks.select("chunk_id", "chunk_text", "slug",
                         "scheme_name", "source_column").collect()

chunk_ids   = [r["chunk_id"]   for r in rows]
chunk_texts = [r["chunk_text"] for r in rows]
chunk_meta  = [{
    "slug":          r["slug"],
    "scheme_name":   r["scheme_name"],
    "source_column": r["source_column"]
} for r in rows]

print(f"✅ {len(chunk_texts)} chunks loaded for embedding")

✅ 18739 chunks loaded for embedding


In [0]:
# CELL 3 — Embed in batches
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

BATCH_SIZE = 64
all_embeddings = []

for i in range(0, len(chunk_texts), BATCH_SIZE):
    batch = chunk_texts[i:i + BATCH_SIZE]
    embs  = model.encode(batch, show_progress_bar=False)
    all_embeddings.append(embs)
    if i % 500 == 0:
        print(f"  Embedded {i}/{len(chunk_texts)}...")

embeddings = np.vstack(all_embeddings).astype("float32")
print(f"✅ Embedding shape: {embeddings.shape}")

/local_disk0/.ephemeral_nfs/envs/pythonEnv-e4e4d829-f1eb-4c56-8d65-1252ad49ed55/lib/python3.12/site-packages/torch/_vmap_internals.py:9: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  from torch.utils._pytree import _broadcast_to_and_flatten, tree_flatten, tree_unflatten


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

  Embedded 0/18739...
  Embedded 8000/18739...
  Embedded 16000/18739...
✅ Embedding shape: (18739, 384)


In [0]:
%sql
CREATE VOLUME IF NOT EXISTS sarkarimitracatalog.sarkaridatabase.sarkari_files;

In [0]:
# CELL 4 — Build FAISS index and save to Unity Catalog Volume
import faiss, json

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)
print(f"✅ FAISS index built. Vectors: {index.ntotal}")

# Unity Catalog Volume path — no DBFS needed
SAVE_DIR = "/Volumes/sarkarimitracatalog/sarkaridatabase/sarkari_files/"

faiss.write_index(index, SAVE_DIR + "faiss_index.bin")

with open(SAVE_DIR + "chunk_id_map.json", "w") as f:
    json.dump({"chunk_ids": chunk_ids, "chunk_meta": chunk_meta}, f)

print("✅ Saved faiss_index.bin and chunk_id_map.json")

✅ FAISS index built. Vectors: 18739
✅ Saved faiss_index.bin and chunk_id_map.json


In [0]:
# CELL 5 — Verify
import os
files = os.listdir("/Volumes/sarkarimitracatalog/sarkaridatabase/sarkari_files/")
print("✅ Files in volume:", files)

✅ Files in volume: ['faiss_index.bin', 'chunk_id_map.json']
